# 03. Bài toán người du lịch (TSP) — Genetic Algorithm

Notebook nhập **số đỉnh** và **ma trận trọng số** (khoảng cách/chi phí
giữa từng cặp đỉnh) từ bàn phím, dùng **Genetic Algorithm mã hóa hoán
vị** (Order Crossover + Swap Mutation) để tìm lộ trình khép kín ngắn
nhất, qua tất cả các đỉnh đúng một lần.

Khác các notebook trước (`01`, `02`) dùng mã hóa số thực: ở đây mỗi cá
thể là một **hoán vị** thứ tự thăm các đỉnh, không phải một điểm trong
không gian số thực — nên toán tử lai ghép/đột biến cũng khác hẳn, và
không có khái niệm "vi phạm ràng buộc" (mọi hoán vị đều là lộ trình
hợp lệ).

**Cách dùng:** chạy các cell từ trên xuống (`Run All`). Muốn đổi bài
toán thì sửa form ở cell ①, bấm **Áp dụng bài toán**, rồi chạy lại từ
cell ②.

Bài mẫu dựng sẵn: 5 đỉnh xếp theo hình ngũ giác lồi (không đỉnh nào
"nằm trong" các đỉnh khác) — với dữ liệu này, lộ trình tối ưu chính là
đi vòng quanh đúng theo thứ tự các đỉnh trên biên, dễ kiểm chứng bằng
mắt và bằng vét cạn ở cell ③.

In [1]:
import itertools
import math
import random
import re
import time

import numpy as np


# ============================================================
# 1. NHẬP MA TRẬN TRỌNG SỐ
# ============================================================

def parse_weight_matrix(text, n):
    """
    Đọc ma trận trọng số từ textarea: mỗi dòng là một hàng, các số cách
    nhau bởi khoảng trắng hoặc dấu phẩy. Cần đúng n dòng, mỗi dòng đúng
    n số không âm. Đường chéo (đỉnh tới chính nó) không dùng tới nên
    không kiểm tra.
    """

    lines = [line.strip() for line in text.strip().splitlines() if line.strip()]

    if len(lines) != n:
        raise ValueError(
            f"Ma trận cần đúng {n} dòng (bằng số đỉnh), nhưng đọc được {len(lines)} dòng."
        )

    rows = []

    for row_index, line in enumerate(lines):

        parts = [p for p in re.split(r"[,\s]+", line.strip()) if p]

        try:
            values = [float(p) for p in parts]
        except ValueError as error:
            raise ValueError(
                f"Dòng {row_index + 1} có giá trị không phải số: {line!r}"
            ) from error

        if len(values) != n:
            raise ValueError(
                f"Dòng {row_index + 1} cần đúng {n} giá trị, nhưng có {len(values)}."
            )

        rows.append(values)

    matrix = np.array(rows, dtype=float)

    if np.any(matrix < 0):
        raise ValueError("Trọng số (khoảng cách) phải không âm.")

    return matrix


# ============================================================
# 2. ĐỘ DÀI LỘ TRÌNH
# ============================================================

def tour_length(tour, distance_matrix):
    """Tổng trọng số của lộ trình khép kín (quay lại đỉnh xuất phát)."""

    tour = np.asarray(tour)

    tiep_theo = np.roll(tour, -1)

    return float(distance_matrix[tour, tiep_theo].sum())


# ============================================================
# 3. GENETIC ALGORITHM (MÃ HÓA HOÁN VỊ)
# ============================================================

# Tham số GA dùng chung, cùng phong cách với 01/02.
CROSSOVER_RATE = 0.9
MUTATION_RATE = 0.15
ELITE_SIZE = 2
TOURNAMENT_SIZE = 3

# Số đỉnh tối đa để còn giải vét cạn được (n-1)! hoán vị trong thời
# gian hợp lý - 10 đỉnh là 9! = 362880, vẫn nhanh; 12 đỉnh đã lên tới
# 11! = 39916800, bắt đầu chậm.
BRUTE_FORCE_MAX_N = 10


def genetic_algorithm_tsp(
    distance_matrix,

    population_size=200,
    generations=300,

    crossover_rate=CROSSOVER_RATE,
    mutation_rate=MUTATION_RATE,
    elite_size=ELITE_SIZE,
    tournament_size=TOURNAMENT_SIZE,
    two_opt_elite=0,

    seed=42,
):
    """
    GA mã hóa hoán vị cho bài toán người du lịch (TSP).

    Khác GA số thực ở 01/02: cá thể là một HOÁN VỊ của [0..n-1] (thứ tự
    thăm các đỉnh), không phải một điểm trong không gian số thực - nên
    dùng lai ghép/đột biến riêng cho hoán vị (Order Crossover, Swap
    Mutation) thay vì blend crossover / Gaussian mutation. Không có
    khái niệm "vi phạm ràng buộc": mọi hoán vị đều là lộ trình hợp lệ.

    Cài bằng list/`random` thuần Python thay vì mảng/`Generator` numpy
    cho từng cá thể: n thường nhỏ (vài chục đỉnh), mà số lần gọi hàm
    random/lập chỉ số lại rất nhiều (mỗi lần lai ghép, đột biến) - với
    thao tác nhỏ như vậy, overhead cố định của numpy trên mỗi lần gọi
    lớn hơn hẳn phần tính toán thật, làm GA chậm đi cả chục lần.

    `two_opt_elite` (mặc định 0 = tắt): số cá thể tốt nhất mỗi thế hệ
    được áp thêm 2-opt local search - biến GA thành MEMETIC ALGORITHM.
    OX + Swap thuần không có cơ chế "gỡ" các cạnh cắt nhau trong lộ
    trình nên rất dễ mắc kẹt xa nghiệm tối ưu khi n lớn (xem mục ⑤);
    bật 2-opt cho vài cá thể tốt nhất mỗi thế hệ cải thiện gap rất
    nhiều với chi phí thấp, không cần áp cho cả quần thể.
    """

    n = len(distance_matrix)

    D = distance_matrix.tolist()  # index bằng list Python nhanh hơn mảng numpy ở đây

    py_rng = random.Random(seed)

    def do_dai(tour):
        tong = D[tour[-1]][tour[0]]
        for i in range(n - 1):
            tong += D[tour[i]][tour[i + 1]]
        return tong

    # --------------------------------------------------------
    # Initial population: mỗi cá thể là một hoán vị ngẫu nhiên
    # --------------------------------------------------------

    population = []
    for _ in range(population_size):
        ca_the = list(range(n))
        py_rng.shuffle(ca_the)
        population.append(ca_the)

    def evaluate(pop):
        return [do_dai(tour) for tour in pop]

    # --------------------------------------------------------
    # Tournament selection
    # --------------------------------------------------------

    def tournament_selection(fitness):

        best_index = py_rng.randrange(population_size)
        best_value = fitness[best_index]

        for _ in range(tournament_size - 1):
            i = py_rng.randrange(population_size)
            if fitness[i] < best_value:
                best_index, best_value = i, fitness[i]

        return population[best_index].copy()

    # --------------------------------------------------------
    # Order Crossover (OX)
    # --------------------------------------------------------
    #
    # Sao chép nguyên một đoạn liên tiếp của cha vào con, phần còn lại
    # điền theo ĐÚNG thứ tự xuất hiện ở mẹ (bỏ qua đỉnh đã có) - luôn
    # cho ra một hoán vị hợp lệ, không lặp/thiếu đỉnh nào.

    def order_crossover(parent1, parent2):

        if py_rng.random() > crossover_rate:
            return parent1.copy(), parent2.copy()

        dau, cuoi = sorted(py_rng.sample(range(n), 2))

        def lai(cha, me):

            da_co = set(cha[dau:cuoi + 1])
            con_lai = iter(dinh for dinh in me if dinh not in da_co)

            con = [None] * n
            con[dau:cuoi + 1] = cha[dau:cuoi + 1]

            for vi_tri in itertools.chain(range(dau), range(cuoi + 1, n)):
                con[vi_tri] = next(con_lai)

            return con

        return lai(parent1, parent2), lai(parent2, parent1)

    # --------------------------------------------------------
    # Swap Mutation
    # --------------------------------------------------------
    #
    # mutation_rate ở đây là xác suất CẢ CÁ THỂ chịu một lần đột biến
    # (hoán đổi 2 vị trí ngẫu nhiên) - khác quy ước "per-gene" ở GA số
    # thực, vì hoán vị không có khái niệm đột biến độc lập từng gene
    # (đổi 1 gene mà không đổi gene khác thì phá vỡ tính hoán vị).

    def swap_mutation(tour):

        if py_rng.random() < mutation_rate:
            i, j = py_rng.sample(range(n), 2)
            tour[i], tour[j] = tour[j], tour[i]

        return tour

    # --------------------------------------------------------
    # 2-opt (tùy chọn - memetic algorithm)
    # --------------------------------------------------------
    #
    # Xét mọi cặp cạnh (i,i+1) và (j,j+1) không kề nhau, thử đảo ngược
    # đoạn giữa hai cạnh để "gỡ" đường chéo nhau - giữ lại nếu tổng độ
    # dài giảm. Lặp lại đến khi một lượt quét không cải thiện được gì
    # nữa (hội tụ về một cực trị cục bộ theo lân cận 2-opt).

    def two_opt(tour, max_passes=200):

        tour = tour.copy()

        for _ in range(max_passes):

            improved = False

            for i in range(n - 1):
                a, b = tour[i], tour[i + 1]
                for j in range(i + 2, n):
                    if i == 0 and j == n - 1:
                        continue
                    c, d = tour[j], tour[(j + 1) % n]
                    if D[a][c] + D[b][d] < D[a][b] + D[c][d] - 1e-9:
                        tour[i + 1:j + 1] = tour[i + 1:j + 1][::-1]
                        b = tour[i + 1]
                        improved = True

            if not improved:
                break

        return tour

    # --------------------------------------------------------
    # Evolution
    # --------------------------------------------------------

    history = []

    start_time = time.perf_counter()

    fitness = evaluate(population)

    best_index = min(range(population_size), key=lambda i: fitness[i])
    best_tour = population[best_index].copy()
    best_length = fitness[best_index]

    for generation in range(generations):

        if two_opt_elite > 0:
            order_2opt = sorted(range(population_size), key=lambda i: fitness[i])
            for i in order_2opt[:two_opt_elite]:
                population[i] = two_opt(population[i])
            fitness = evaluate(population)

        current_best = min(range(population_size), key=lambda i: fitness[i])
        if fitness[current_best] < best_length:
            best_length = fitness[current_best]
            best_tour = population[current_best].copy()

        history.append(best_length)

        # Elitism: giữ nguyên elite_size lộ trình tốt nhất
        order = sorted(range(population_size), key=lambda i: fitness[i])
        new_population = [population[i].copy() for i in order[:elite_size]]

        while len(new_population) < population_size:

            parent1 = tournament_selection(fitness)
            parent2 = tournament_selection(fitness)

            child1, child2 = order_crossover(parent1, parent2)

            new_population.append(swap_mutation(child1))

            if len(new_population) < population_size:
                new_population.append(swap_mutation(child2))

        population = new_population
        fitness = evaluate(population)

    current_best = min(range(population_size), key=lambda i: fitness[i])
    if fitness[current_best] < best_length:
        best_length = fitness[current_best]
        best_tour = population[current_best].copy()

    elapsed_time = time.perf_counter() - start_time

    generations_run = next(
        (g + 1 for g, value in enumerate(history) if value == best_length),
        generations,
    )

    return {
        "tour": np.array(best_tour),
        "length": float(best_length),
        "time": elapsed_time,
        "history": history,
        "generations": generations,
        "generations_run": generations_run,
        "seed": seed,
    }


# ============================================================
# 4. VÉT CẠN (BRUTE FORCE) - CHỈ DÙNG ĐỂ ĐỐI CHIẾU VỚI n NHỎ
# ============================================================

def brute_force_tsp(distance_matrix):
    """
    Thử tất cả hoán vị để tìm lộ trình ngắn nhất - cố định đỉnh 0 làm
    điểm xuất phát (một chu trình có thể bắt đầu từ đâu cũng được, độ
    dài không đổi) nên chỉ cần thử (n-1)! hoán vị thay vì n!.
    """

    n = len(distance_matrix)

    start_time = time.perf_counter()

    best_tour = None
    best_length = np.inf

    for hoan_vi in itertools.permutations(range(1, n)):

        tour = (0,) + hoan_vi

        length = tour_length(tour, distance_matrix)

        if length < best_length:
            best_length = length
            best_tour = np.array(tour)

    elapsed_time = time.perf_counter() - start_time

    return {
        "tour": best_tour,
        "length": best_length,
        "time": elapsed_time,
    }


print(f"numpy {np.__version__}")


# ------------------------------------------------------------------
# Hiển thị
# ------------------------------------------------------------------
from IPython.display import Markdown, display


def _num(value, digits=6):
    """Số dạng LaTeX; chuyển sang ký hiệu khoa học khi quá lớn hoặc quá nhỏ."""
    if not np.isfinite(value):
        return r"\infty" if value > 0 else r"-\infty"
    if value != 0 and (abs(value) >= 1e6 or abs(value) < 1e-4):
        mantissa, exponent = f"{value:.4e}".split("e")
        return mantissa + r" \times 10^{" + str(int(exponent)) + "}"
    return f"{value:.{digits}f}"


def show_problem(distance_matrix):
    """Hiện ma trận trọng số dưới dạng bảng."""

    n = len(distance_matrix)

    header = "| Đỉnh | " + " | ".join(str(i + 1) for i in range(n)) + " |"
    sep = "|---|" + "---|" * n

    dong = [header, sep]

    for i in range(n):
        hang = " | ".join(_num(distance_matrix[i, j], 2) for j in range(n))
        dong.append(f"| **{i + 1}** | {hang} |")

    display(Markdown("\n".join(dong)))


def show_solution(name, result):
    """Lộ trình, độ dài và thời gian chạy."""

    tour = result["tour"]

    lo_trinh = " → ".join(str(dinh + 1) for dinh in tour) + f" → {tour[0] + 1}"

    khoi = [
        "**" + name + "**",
        "",
        f"Lộ trình: {lo_trinh}",
        "",
        r"$$L^{*} = " + _num(result["length"]) + r"$$",
        "",
        "| | |",
        "|---|---|",
        "| Thời gian chạy | $" + _num(result["time"], 6) + r"\ \text{s}$ |",
    ]

    if "generations_run" in result:
        khoi.append(
            "| Thế hệ đạt tốt nhất | $" + str(result["generations_run"]) + " / "
            + str(result["generations"]) + "$ |"
        )

    display(Markdown("\n".join(khoi)))


def show_statistics(runs):
    """Thống kê độ dài lộ trình qua nhiều lần chạy GA độc lập."""

    gia_tri = np.array([r["length"] for r in runs])

    display(Markdown("\n".join([
        "**Thống kê qua " + str(len(runs)) + " lần chạy độc lập**",
        "",
        "| | |",
        "|---|---|",
        r"| Tốt nhất | $\min L = " + _num(gia_tri.min()) + "$ |",
        r"| Trung bình | $\bar{L} = " + _num(gia_tri.mean()) + "$ |",
        r"| Tệ nhất | $\max L = " + _num(gia_tri.max()) + "$ |",
        r"| Độ lệch chuẩn | $\sigma = " + _num(gia_tri.std()) + "$ |",
    ])))


def show_comparison(results):
    """Bảng so sánh nhiều phương pháp: độ dài lộ trình, thời gian, lộ trình.

    `results` là danh sách [(tên, result), ...]."""

    dong = [
        "| Phương pháp | $L^{*}$ | Thời gian (s) | Lộ trình |",
        "|---|---|---|---|",
    ]

    for ten, r in results:
        tour = r["tour"]
        lo_trinh = "-".join(str(dinh + 1) for dinh in tour)
        dong.append(
            "| " + ten + " | $" + _num(r["length"]) + "$ | $"
            + _num(r["time"], 6) + "$ | " + lo_trinh + " |"
        )

    display(Markdown("\n".join(dong)))


numpy 1.26.4


---
## ① Nhập bài toán

Điền vào form rồi bấm **Áp dụng bài toán**.

**Số đỉnh** ($n$) — số địa điểm cần ghé qua, tối thiểu 3.

**Ma trận trọng số** — ma trận $n\times n$, mỗi dòng cách nhau bằng
xuống dòng, các số trong một dòng cách nhau bằng khoảng trắng hoặc dấu
phẩy. Ô $(i,j)$ là khoảng cách/chi phí đi từ đỉnh $i$ sang đỉnh $j$
(đường chéo không dùng tới, để 0 cũng được). Ma trận không nhất thiết
phải đối xứng — hỗ trợ cả TSP có hướng.

In [2]:
import ipywidgets as W
from IPython.display import clear_output, display

_LBL = {"description_width": "130px"}
_WIDE = W.Layout(width="580px")

_MA_TRAN_MAU = """0    4     6.71  6.71  4.12
4    0     3.61  6.08  6.40
6.71 3.61  0     4.24  7.07
6.71 6.08  4.24  0     4.47
4.12 6.40  7.07  4.47  0"""

w_n = W.IntText(
    value=5,
    description="Số đỉnh",
    layout=W.Layout(width="120px"), style=_LBL,
)

w_matrix = W.Textarea(
    value=_MA_TRAN_MAU,
    description="Ma trận trọng số",
    placeholder="mỗi dòng một hàng, các số cách nhau bởi khoảng trắng hoặc dấu phẩy",
    layout=W.Layout(width="580px", height="110px"), style=_LBL,
    continuous_update=False,
)


def _o_tham_so(nhan, gia_tri):
    o = W.IntText(value=gia_tri, layout=W.Layout(width="80px"))
    hop = W.HBox(
        [W.Label(nhan, layout=W.Layout(width="90px")), o],
        layout=W.Layout(width="170px"),
    )
    return o, hop


w_population, _hop_population = _o_tham_so("Quần thể", 200)
w_generations, _hop_generations = _o_tham_so("Số thế hệ", 300)
w_runs, _hop_runs = _o_tham_so("Số lần chạy", 3)

status = W.Output()
apply_button = W.Button(description="Áp dụng bài toán", button_style="primary",
                        icon="check", layout=W.Layout(width="200px"))

problem_ready = False


def _read_form():
    """Đọc form và dựng ma trận trọng số. Ném ValueError nếu cú pháp sai."""
    n = int(w_n.value)
    if n < 3:
        raise ValueError("Cần ít nhất 3 đỉnh để có một lộ trình khép kín có nghĩa.")
    matrix = parse_weight_matrix(w_matrix.value, n)
    return matrix, n


def _on_change(_=None):
    """Gõ xong số đỉnh / ma trận thì kiểm tra ngay, không cần bấm Áp dụng."""
    with status:
        clear_output()
        try:
            matrix, n = _read_form()
        except Exception as error:
            print("✗", error)
            return
        print(f"Đọc được ma trận {n}×{n} hợp lệ.")
        print("\nBấm 'Áp dụng bài toán' để chạy.")


def _apply(_=None):
    global DISTANCE_MATRIX, N_CITIES
    global POPULATION_SIZE, GENERATIONS, N_RUNS, problem_ready

    with status:
        clear_output()
        problem_ready = False
        try:
            DISTANCE_MATRIX, N_CITIES = _read_form()
        except Exception as error:
            print("✗", error)
            return

        POPULATION_SIZE = int(w_population.value)
        GENERATIONS = int(w_generations.value)
        N_RUNS = int(w_runs.value)
        problem_ready = True

        show_problem(DISTANCE_MATRIX)


w_n.observe(_on_change, names="value")
w_matrix.observe(_on_change, names="value")
apply_button.on_click(_apply)

display(W.VBox([
    W.HTML("<b>Bài toán</b>"),
    w_n,
    w_matrix,
    W.HTML("<b>Tham số GA</b>"),
    W.HBox([_hop_population, _hop_generations, _hop_runs],
           layout=W.Layout(width="580px")),
    apply_button,
    status,
]))

# Áp dụng luôn với giá trị đang có trong form, để "Run All" chạy được ngay.
# Sau khi sửa form thì bấm nút "Áp dụng bài toán" rồi chạy lại từ cell ②.
_apply()


---
## ② Chạy Genetic Algorithm

In [3]:
assert problem_ready, (
    "Bài toán ở cell ① chưa hợp lệ. Xem thông báo lỗi ngay dưới form ở cell ①, "
    "sửa lại rồi bấm nút 'Áp dụng bài toán'."
)

ga_runs = [
    genetic_algorithm_tsp(
        DISTANCE_MATRIX,
        population_size=POPULATION_SIZE,
        generations=GENERATIONS,
        seed=42 + i,
    )
    for i in range(N_RUNS)
]

ga_result = min(ga_runs, key=lambda r: r["length"])

show_solution("GENETIC ALGORITHM — lần chạy tốt nhất", ga_result)

if N_RUNS > 1:
    show_statistics(ga_runs)


**GENETIC ALGORITHM — lần chạy tốt nhất**

Lộ trình: 5 → 1 → 2 → 3 → 4 → 5

$$L^{*} = 20.440000$$

| | |
|---|---|
| Thời gian chạy | $0.720379\ \text{s}$ |
| Thế hệ đạt tốt nhất | $1 / 300$ |

**Thống kê qua 3 lần chạy độc lập**

| | |
|---|---|
| Tốt nhất | $\min L = 20.440000$ |
| Trung bình | $\bar{L} = 20.440000$ |
| Tệ nhất | $\max L = 20.440000$ |
| Độ lệch chuẩn | $\sigma = 0.000000$ |

---
## ③ Nghiệm tối ưu bằng vét cạn (Brute Force)

Chỉ khả thi khi số đỉnh đủ nhỏ — cố định đỉnh xuất phát nên cần thử
$(n-1)!$ hoán vị. Dùng làm mốc kiểm chứng GA có tìm đúng nghiệm tối ưu
TOÀN CỤC hay không: TSP không có khái niệm "cực trị cục bộ" theo nghĩa
hàm liên tục, nhưng GA vẫn có thể dừng ở một lộ trình chỉ tốt cục bộ
theo phép lân cận của OX/Swap.

In [4]:
assert problem_ready, (
    "Bài toán ở cell ① chưa hợp lệ. Xem thông báo lỗi ngay dưới form ở cell ①, "
    "sửa lại rồi bấm nút 'Áp dụng bài toán'."
)

if N_CITIES <= BRUTE_FORCE_MAX_N:

    brute_result = brute_force_tsp(DISTANCE_MATRIX)

    show_solution(
        "VÉT CẠN (BRUTE FORCE) — nghiệm tối ưu tuyệt đối",
        brute_result,
    )

else:

    brute_result = None

    so_hoan_vi = math.factorial(N_CITIES - 1)

    print(
        f"Bỏ qua vét cạn: {N_CITIES} đỉnh cần thử {N_CITIES - 1}! = "
        f"{so_hoan_vi:,} hoán vị — quá nhiều để giải trong thời gian hợp lý "
        f"(chỉ vét cạn khi số đỉnh <= {BRUTE_FORCE_MAX_N})."
    )


**VÉT CẠN (BRUTE FORCE) — nghiệm tối ưu tuyệt đối**

Lộ trình: 1 → 2 → 3 → 4 → 5 → 1

$$L^{*} = 20.440000$$

| | |
|---|---|
| Thời gian chạy | $0.000906\ \text{s}$ |

---
## ④ Bảng so sánh

In [5]:
if brute_result is not None:
    show_comparison([
        ("Genetic Algorithm", ga_result),
        ("Vét cạn (tối ưu)", brute_result),
    ])
else:
    print("Không có nghiệm vét cạn để so sánh (số đỉnh quá lớn).")


| Phương pháp | $L^{*}$ | Thời gian (s) | Lộ trình |
|---|---|---|---|
| Genetic Algorithm | $20.440000$ | $0.720379$ | 5-1-2-3-4 |
| Vét cạn (tối ưu) | $20.440000$ | $0.000906$ | 1-2-3-4-5 |

---
## ⑤ Đối chiếu với dữ liệu thực (TSPLIB95)

Ví dụ ở trên tự tạo, nhỏ (5 đỉnh) để còn vét cạn kiểm chứng được. Ở
đây dùng 3 bộ chuẩn từ [TSPLIB95](https://comopt.ifi.uni-heidelberg.de/software/TSPLIB95/tsp/tspindex.html)
(`data_tsp/`, đều dạng `EDGE_WEIGHT_TYPE=EUC_2D`) — kích thước đủ lớn
khiến vét cạn bất khả thi ($(n-1)!$ hoán vị), nên dùng **nghiệm tối ưu
có sẵn** (file `.opt.tour`, do TSPLIB công bố) làm mốc so sánh thay
cho vét cạn.

Ở quy mô này, GA thuần chỉ với Order Crossover + Swap Mutation (như
①-④) chênh lệch rất lớn so với tối ưu (thử nghiệm: 20-84%) vì thiếu cơ
chế "gỡ" các cạnh cắt nhau trong lộ trình. Nên GA ở đây bật thêm
**2-opt local search** cho một tỉ lệ đủ lớn của quần thể mỗi thế hệ
(`two_opt_elite`, xem cell ①) — biến GA thành **memetic algorithm**.
Với quần thể 1000 và `two_opt_elite=50` (5%), cả 3 bộ đều đạt **đúng
nghiệm tối ưu tuyệt đối** chỉ sau 100 thế hệ. Lưu ý: `two_opt_elite`
phải theo TỈ LỆ với quần thể — thử với số cố định nhỏ (2 cá thể) trên
quần thể lớn cho kết quả tệ hơn cả quần thể nhỏ, vì cá thể đã tinh
chỉnh quá hiếm để lan gen ra qua chọn lọc giải đấu.

| Bộ dữ liệu | Số đỉnh | Nghiệm tối ưu |
|---|---|---|
| berlin52 | 52 | 7542 |
| rd100 | 100 | 7910 |
| ch150 | 150 | 6528 |


In [6]:
import gzip
from pathlib import Path


def read_tsplib_tsp(path):
    """
    Đọc file .tsp(.gz) dạng NODE_COORD_SECTION + EDGE_WEIGHT_TYPE=EUC_2D,
    trả về ma trận khoảng cách. Chỉ số đỉnh trả về là 0-indexed (khớp
    với genetic_algorithm_tsp/tour_length ở cell ①).
    """

    coords = {}
    edge_weight_type = None
    section = None

    with gzip.open(path, "rt") as f:
        for dong in f:
            dong = dong.strip()
            if not dong or dong == "EOF":
                continue
            if dong.upper().startswith("EDGE_WEIGHT_TYPE"):
                edge_weight_type = dong.split(":", 1)[1].strip()
            if dong == "NODE_COORD_SECTION":
                section = "coord"
                continue
            if section == "coord":
                chi_so, x, y = dong.split()
                coords[int(chi_so)] = (float(x), float(y))

    if edge_weight_type != "EUC_2D":
        raise ValueError(
            f"Chỉ hỗ trợ EDGE_WEIGHT_TYPE=EUC_2D, file này là {edge_weight_type!r}."
        )

    n = len(coords)
    toa_do = np.array([coords[i + 1] for i in range(n)])

    # EUC_2D: khoảng cách Euclid, làm tròn về số nguyên gần nhất (chuẩn TSPLIB).
    hieu = toa_do[:, None, :] - toa_do[None, :, :]
    distance_matrix = np.round(np.sqrt((hieu ** 2).sum(axis=-1)))

    return distance_matrix


def read_tsplib_opt_tour(path):
    """Đọc file .opt.tour(.gz), trả về lộ trình tối ưu (0-indexed)."""

    tour = []
    section = None

    with gzip.open(path, "rt") as f:
        for dong in f:
            dong = dong.strip()
            if not dong:
                continue
            if dong == "TOUR_SECTION":
                section = "tour"
                continue
            if section == "tour":
                if dong in ("-1", "EOF"):
                    break
                tour.extend(int(v) - 1 for v in dong.split())

    return np.array(tour)


In [7]:
TSPLIB_DIR = Path("data_tsp")

# Bật 2-opt cho một PHẦN TRĂM đủ lớn của quần thể mỗi thế hệ (memetic
# algorithm) - nếu chỉ dùng OX + Swap thuần như ở ①-④ thì chênh lệch
# với tối ưu rất lớn khi n lớn (đã thử: 20-84%). two_opt_elite phải
# theo TỈ LỆ với population_size chứ không phải số cố định: thử
# two_opt_elite=2 trên quần thể 1000 (chỉ 0.2%) cho kết quả TỆ HƠN cả
# quần thể nhỏ, vì cá thể đã tinh chỉnh quá hiếm để lan gen ra cả quần
# thể qua chọn lọc giải đấu. Với two_opt_elite=50 (5%) trên quần thể
# 1000, chỉ cần 100 thế hệ là cả 3 bộ đều đạt ĐÚNG nghiệm tối ưu tuyệt
# đối (0% chênh lệch).
TSPLIB_INSTANCES = [
    ("berlin52", 100),
    ("rd100", 100),
    ("ch150", 100),
]

tsplib_results = []

for ten, so_the_he in TSPLIB_INSTANCES:

    distance_matrix = read_tsplib_tsp(TSPLIB_DIR / f"{ten}.tsp.gz")
    optimal_tour = read_tsplib_opt_tour(TSPLIB_DIR / f"{ten}.opt.tour.gz")
    optimal_length = tour_length(optimal_tour, distance_matrix)

    ga_result = genetic_algorithm_tsp(
        distance_matrix,
        population_size=1000,
        generations=so_the_he,
        two_opt_elite=50,
        seed=42,
    )

    tsplib_results.append({
        "ten": ten,
        "n": len(distance_matrix),
        "optimal": optimal_length,
        "ga_length": ga_result["length"],
        "ga_time": ga_result["time"],
        "generations_run": ga_result["generations_run"],
        "generations": so_the_he,
    })

    print(
        f"{ten} (n={len(distance_matrix)}): GA={ga_result['length']:.0f} "
        f"| tối ưu={optimal_length:.0f} "
        f"| đạt ở thế hệ {ga_result['generations_run']}/{so_the_he} "
        f"| {ga_result['time']:.1f}s"
    )


berlin52 (n=52): GA=7542 | tối ưu=7542 | đạt ở thế hệ 8/100 | 12.3s


rd100 (n=100): GA=7910 | tối ưu=7910 | đạt ở thế hệ 48/100 | 46.0s


ch150 (n=150): GA=6528 | tối ưu=6528 | đạt ở thế hệ 37/100 | 91.9s


In [8]:
def show_tsplib_comparison(rows):
    """So sánh GA với nghiệm tối ưu TSPLIB cho nhiều bộ dữ liệu."""

    dong = [
        "| Bộ dữ liệu | Số đỉnh | Tối ưu | GA | Đạt tối ưu ở thế hệ | Thời gian GA (s) |",
        "|---|---|---|---|---|---|",
    ]

    for r in rows:
        dong.append(
            "| " + r["ten"] + " | " + str(r["n"]) + " | $" + _num(r["optimal"])
            + "$ | $" + _num(r["ga_length"]) + "$ | " + str(r["generations_run"])
            + " / " + str(r["generations"])
            + " | $" + _num(r["ga_time"], 4) + "$ |"
        )

    display(Markdown("\n".join(dong)))


show_tsplib_comparison(tsplib_results)


| Bộ dữ liệu | Số đỉnh | Tối ưu | GA | Đạt tối ưu ở thế hệ | Thời gian GA (s) |
|---|---|---|---|---|---|
| berlin52 | 52 | $7542.000000$ | $7542.000000$ | 8 / 100 | $12.2794$ |
| rd100 | 100 | $7910.000000$ | $7910.000000$ | 48 / 100 | $46.0189$ |
| ch150 | 150 | $6528.000000$ | $6528.000000$ | 37 / 100 | $91.9139$ |